# Group XX - Financial Data Science: Market Analysis
**University of Kelaniya | Department of Finance | Applied Financial Data Analytics and Decision Reporting Using Python**

Group: **XX** &nbsp;|&nbsp; Due: 31.10.2026

> **How to use this notebook:** search for `TODO(PROMPT)` (code cells) and `TODO:` (Markdown cells).
> Each one tells you exactly what to fill in or what to ask Claude once the dataset is added.
> When everything is done: **Kernel -> Restart & Run All**, and make sure no cell errors.

---
## Task 1 - Decision Brief and Data Plan

### 1.1 Decision brief (250-350 words)

> **TODO: PROMPT for Claude (after the dataset is added):**
> *"Here are my four assets, the benchmark, the source and the period: [paste]. Write a 250-350 word decision brief covering:
> (a) who uses the analysis (Investment Committee of an investment research firm), (b) the decision it supports
> (which assets need closer monitoring), (c) the evidence the notebook will produce, (d) why each asset and the benchmark
> are relevant, and (e) how this workflow applies Financial Data Science to a real financial decision.
> Plain language, no personal investment advice."*
>
> Delete this TODO box and paste the final brief here. Keep it within 250-350 words (check with a word counter).

### 1.2 Selected assets and benchmark

| Role | Name | Ticker / column in raw file | Why it is relevant to the decision |
|---|---|---|---|
| Asset 1 | TODO | TODO | TODO |
| Asset 2 | TODO | TODO | TODO |
| Asset 3 | TODO | TODO | TODO |
| Asset 4 | TODO | TODO | TODO |
| Benchmark | TODO | TODO | TODO |

> **TODO: PROMPT:** *"Justify in 1-2 sentences each why [asset] is relevant to an Investment Committee market-monitoring briefing (sector, size, liquidity, role vs the benchmark)."*

### 1.3 Three practical questions

1. **Performance:** TODO (e.g. Which asset delivered the highest and lowest total return, and did each beat the benchmark?)
2. **Risk:** TODO (e.g. Which asset had the highest daily volatility, and which had the most extreme or skewed daily moves?)
3. **Co-movement / diversification:** TODO (e.g. Which pairs of assets have the lowest correlation, and how closely does each follow the benchmark?)

### 1.4 Data dictionary

| Item | Description |
|---|---|
| Data source | TODO (exact source name, e.g. CSE data library / Yahoo Finance via yfinance / lecturer-approved dataset) |
| Retrieval date | TODO (set the same value in `RETRIEVAL_DATE` in the configuration cell) |
| Observation period | TODO (first date to last date; at least 24 months) |
| Frequency | Daily (trading days) |
| Variables | `Date`; one price column per asset and for the benchmark |
| Price field used | TODO (e.g. Adjusted Close; justify in Task 3) |
| Units | TODO (e.g. LKR per share / USD per share / index points). Returns are unitless decimals (0.01 = 1%) |
| Known limitations | TODO (e.g. different holiday calendars, thin trading, survivorship, dividends not included if unadjusted Close, short period) |

---
## Task 2 - Reproducible Data Pipeline and Coding

### 2.1 Configuration
All user-adjustable settings are in this one cell. Paths are relative (`pathlib`), so any analyst can run the notebook from the project folder.

In [ ]:
# ===================== CONFIGURATION (market-monitoring briefing pipeline) =====================
from pathlib import Path
from datetime import date
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.float_format = "{:,.6f}".format

# Project root works whether Jupyter starts in notebook/ or in the root folder
ROOT = Path.cwd().resolve()
if ROOT.name == "notebook":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import finance_tools as ft

# TODO(PROMPT): Replace the 4 asset names/tickers and the benchmark after the dataset is added.
# The KEY is the name shown in tables/figures; the VALUE must match the ticker (yfinance) or the
# column header in raw_market_data.csv.
# Ask Claude: "My raw file columns are: [paste header row]. Fill in ASSETS and BENCHMARK."
ASSETS = {
    "Asset 1": "TICKER_1",
    "Asset 2": "TICKER_2",
    "Asset 3": "TICKER_3",
    "Asset 4": "TICKER_4",
}
BENCHMARK = {"Benchmark": "BENCHMARK_TICKER"}

# TODO(PROMPT): "csv" if you were given/downloaded a file (put it at data/raw/raw_market_data.csv),
# "yfinance" if the notebook should download it. Ask Claude if your file is not wide-format.
DATA_SOURCE = "csv"
START_DATE = "2024-01-01"   # TODO: first date (at least 24 months before END_DATE)
END_DATE = "2026-09-30"     # TODO: last date
PRICE_FIELD = "Adj Close"   # TODO: "Adj Close" or "Close"; justify in Task 3
SOURCE_NAME = "TODO: exact data source"      # appears on every figure
RETRIEVAL_DATE = "TODO: YYYY-MM-DD"          # the day the data were downloaded

MAX_FFILL_DAYS = 3          # gaps up to 3 trading days are forward-filled (holidays); logged
EXTREME_THRESHOLD = 0.10    # flag daily moves larger than +/-10%
MIN_MONTHS = 24

# Folders and output filenames
RAW_FILE = ROOT / "data" / "raw" / "raw_market_data.csv"
CLEAN_FILE = ROOT / "data" / "processed" / "clean_financial_data.csv"
OUT_DIR = ROOT / "outputs"
FIG_DIR = OUT_DIR / "figures"
AUDIT_FILE = OUT_DIR / "data_audit.csv"
QUALITY_LOG_FILE = OUT_DIR / "data_quality_log.csv"
SUMMARY_FILE = OUT_DIR / "summary_statistics.csv"
COV_FILE = OUT_DIR / "covariance_matrix.csv"
CORR_FILE = OUT_DIR / "correlation_matrix.csv"

ALL_ASSETS = {**ASSETS, **BENCHMARK}                       # display name -> ticker
TICKER_TO_NAME = {v: k for k, v in ALL_ASSETS.items()}
print("Project root:", ROOT)
print("Assets      :", list(ASSETS), "| Benchmark:", list(BENCHMARK))

### 2.2 Acquire the data and preserve an unchanged raw copy
- `csv` mode: the file you place in `data/raw/` is the raw copy. It is **never overwritten**; a SHA-256 fingerprint is recorded so reviewers can verify it is unchanged.
- `yfinance` mode: the file is downloaded **once** and saved untouched; later runs reuse it.

In [ ]:
# Acquire data (market-monitoring briefing source) and keep the raw file untouched
RAW_FILE.parent.mkdir(parents=True, exist_ok=True)

if DATA_SOURCE == "yfinance":
    if RAW_FILE.exists():
        print("Raw file already exists - kept unchanged:", RAW_FILE.name)
    else:
        raw_download = ft.download_yfinance(list(ALL_ASSETS.values()), START_DATE, END_DATE)
        raw_download.to_csv(RAW_FILE, index=False)
        print("Downloaded and saved raw copy:", RAW_FILE.name)
elif DATA_SOURCE == "csv":
    if not RAW_FILE.exists():
        raise FileNotFoundError(f"Put the approved dataset at {RAW_FILE.relative_to(ROOT)}")
else:
    raise ValueError("DATA_SOURCE must be 'csv' or 'yfinance'")

RAW_SHA256 = ft.file_sha256(RAW_FILE)
print("Raw file :", RAW_FILE.relative_to(ROOT))
print("SHA-256  :", RAW_SHA256)
print("Retrieval date:", RETRIEVAL_DATE)

raw = ft.load_data(RAW_FILE)   # TODO(PROMPT): if your file has a different date column name, pass date_col="..."

### 2.3 First look at the raw data

In [ ]:
print("Shape (rows, columns):", raw.shape)
print("Date range:", raw["Date"].min().date(), "to", raw["Date"].max().date())
print("\nData types:")
print(raw.dtypes)
print("\nMissing values per column:")
print(raw.isna().sum())
print("\nFirst 5 observations:")
display(raw.head())
print("Last 5 observations:")
display(raw.tail())

In [ ]:
# Data-audit table (required deliverable) - also checks the 24-month minimum
audit = ft.validate_data(raw, min_months=MIN_MONTHS)
ft.save_outputs(audit, AUDIT_FILE)
display(audit)

**Interpretation of the raw-data audit**

> **TODO: PROMPT for Claude:** *"This is my data-audit table: [paste]. In 3-4 sentences state: how many observations and columns there are, the period covered, which columns have missing values and how many, and what that means for the next cleaning step."*

---
## Task 3 - Data Quality and Feature Engineering

### 3.1 Select the price field and standardise the structure
**Decision (TODO):** explain which price field is used and why.

> **TODO: PROMPT:** *"Explain in 3-4 sentences why [Adjusted Close / Close] is the right price field for daily simple returns for [my assets], mentioning dividends, splits and rights issues."* (Adjusted Close is usually preferred because it accounts for splits and dividends; if your source has only Close, state that as a limitation.)

In [ ]:
# Select the analytical price field; columns are renamed from ticker -> display name
prices_raw = ft.select_price_field(raw, PRICE_FIELD, list(ALL_ASSETS.values()))
prices_raw = prices_raw.rename(columns=TICKER_TO_NAME)
print("Columns used for analysis:", [c for c in prices_raw.columns if c != "Date"])
display(prices_raw.head())

### 3.2 Clean dates, numbers, order, duplicates and missing values
`clean_prices()` performs these steps and writes **every** action to the quality log (nothing is changed silently):

| Step | Rule | Justification |
|---|---|---|
| Dates | Unparsable dates -> row removed | A price without a date cannot be placed in the time series |
| Numbers | Non-numeric or zero/negative prices -> missing | Prices cannot be <= 0 |
| Order | Sort ascending by date | Returns need chronological order |
| Duplicates | Keep the last record per date | The later record is normally the corrected one |
| Missing (short) | Forward-fill up to 3 trading days | Holidays/halts: the last traded price remains the market's latest valuation |
| Missing (long/leading) | Remove the row | Filling long gaps would create artificial zero returns |

> **TODO: PROMPT:** *"After seeing my quality log: [paste counts], rewrite this table so it only describes decisions that actually happened in my data, and explain why forward-fill (not deletion or interpolation) was used for the specific gaps."*

In [ ]:
# Clean the market-monitoring briefing prices and keep a full log
prices, quality_log = ft.clean_prices(prices_raw, max_ffill=MAX_FFILL_DAYS)

print("Rows before cleaning:", len(prices_raw), "| after cleaning:", len(prices))
print("Log entries:", len(quality_log))
if len(quality_log):
    display(quality_log.groupby(["issue", "action"]).size().rename("count").reset_index())
else:
    print("No cleaning actions were needed.")

### 3.3 Daily simple returns, normalised index and extreme moves
$$Return_t = \frac{Price_t}{Price_{t-1}} - 1 \qquad \text{Index}_t = 100 \times \frac{Price_t}{Price_0}$$

In [ ]:
returns = ft.calculate_returns(prices)
normalized = ft.normalize_prices(prices, base=100.0)

extremes = ft.flag_extreme_moves(returns, EXTREME_THRESHOLD)
print(f"Daily moves larger than +/-{EXTREME_THRESHOLD:.0%}: {len(extremes)}")
display(extremes)

**Extreme price-change investigation**

> **TODO: PROMPT for Claude:** *"My flagged extreme moves are: [paste the table above]. For each one, search for what happened on that date and help me classify it as (a) valid market movement, (b) corporate action (split, dividend, rights issue, bonus issue) or (c) data error. Give a source link for each and one sentence of justification."*
>
> Then fill the dictionary in the next cell. Data are **never** manually edited; if a move is a data error, document it and discuss the limitation (or re-download from the source).

In [ ]:
# TODO(PROMPT): After investigating each flagged move, record the decision here, e.g.
# EXTREME_DECISIONS = {("2025-03-04", "Asset 1"): "Valid market movement - earnings announcement (source: ...)"}
EXTREME_DECISIONS = {}

extra_rows = []
for _, row in extremes.iterrows():
    key = (row["date"].strftime("%Y-%m-%d"), row["asset"])
    decision = EXTREME_DECISIONS.get(key, "NOT YET CLASSIFIED - investigate before submission")
    extra_rows.append({"date": row["date"], "column": row["asset"],
                       "issue": f"extreme daily move ({row['daily_return']:+.1%})",
                       "action": f"retained, no value altered; decision: {decision}"})
quality_log_full = pd.concat([quality_log, pd.DataFrame(extra_rows)], ignore_index=True)
ft.save_outputs(quality_log_full, QUALITY_LOG_FILE, index=False)
print("Quality log rows:", len(quality_log_full))
unclassified = [r for r in extra_rows if "NOT YET CLASSIFIED" in r["action"]]
print("Extreme moves still to classify:", len(unclassified))

### 3.4 Automated validation checks

In [ ]:
# Visible validation results (assertions raise an error if any check fails)
validation = ft.run_quality_checks(prices, returns)
display(validation)

# Extra explicit assertions required by the brief
assert prices.index.is_unique, "Duplicate dates found"
assert prices.index.is_monotonic_increasing, "Dates are not in ascending order"
assert not prices.isna().any().any(), "Missing analytical values remain"
print("All data-quality assertions passed.")

In [ ]:
# Save the cleaned dataset: prices + daily returns + normalised index in one table
clean = pd.concat(
    [prices.add_suffix("_price"),
     returns.add_suffix("_return"),
     normalized.add_suffix("_index100")], axis=1)
clean.index.name = "Date"
ft.save_outputs(clean, CLEAN_FILE)

# Output-file creation check
display(ft.check_files_exist([AUDIT_FILE, QUALITY_LOG_FILE, CLEAN_FILE]))
display(clean.head())

**Justification of the main cleaning decisions**

> **TODO: PROMPT for Claude:** *"Using my quality log counts [paste] and the extreme-move classifications [paste], write 150-200 words justifying the main cleaning decisions (price field, duplicates, missing-value treatment, extreme moves) and stating the effect on the analysis."*

---
## Task 4 - Statistical Analysis and Financial Interpretation

### 4.1 Summary statistics (daily return series)

In [ ]:
summary = ft.summary_statistics(prices, returns)
ft.save_outputs(summary, SUMMARY_FILE)
display(summary)

**Interpretation (Observation - numerical evidence - financial implication - limitation)**

> **TODO: PROMPT for Claude:** *"Here is my summary_statistics table: [paste]. Write the interpretation in this exact sequence: Observation, Numerical evidence, Financial implication, Limitation. Identify the strongest and weakest performer (total period return), highest and lowest volatility (std. dev.), unusual distribution characteristics (skewness, excess kurtosis), and how each asset compares with the benchmark. Keep every number identical to the table."*

In [ ]:
# Quick facts to help write the interpretation (all derived from the tables above)
assets_only = list(ASSETS)
bench = list(BENCHMARK)[0]
print("Best total return   :", summary.loc[assets_only, "Total period return"].idxmax(),
      f"{summary.loc[assets_only, 'Total period return'].max():.2%}")
print("Worst total return  :", summary.loc[assets_only, "Total period return"].idxmin(),
      f"{summary.loc[assets_only, 'Total period return'].min():.2%}")
print("Highest volatility  :", summary.loc[assets_only, "Standard deviation"].idxmax(),
      f"{summary.loc[assets_only, 'Standard deviation'].max():.4f}")
print("Lowest volatility   :", summary.loc[assets_only, "Standard deviation"].idxmin(),
      f"{summary.loc[assets_only, 'Standard deviation'].min():.4f}")
print(f"Benchmark total return: {summary.loc[bench, 'Total period return']:.2%}")
print("Assets beating benchmark:",
      [a for a in assets_only if summary.loc[a, "Total period return"] > summary.loc[bench, "Total period return"]])
print("Most negatively skewed :", summary.loc[assets_only, "Skewness"].idxmin())
print("Heaviest tails (kurtosis):", summary.loc[assets_only, "Kurtosis (excess)"].idxmax())

### 4.2 Covariance and correlation matrices

In [ ]:
cov_matrix = returns.cov()
corr_matrix = returns.corr()
ft.save_outputs(cov_matrix, COV_FILE)
ft.save_outputs(corr_matrix, CORR_FILE)
print("Covariance matrix (daily returns):")
display(cov_matrix)
print("Correlation matrix (daily returns):")
display(corr_matrix)

In [ ]:
# Asset pairs ranked by correlation (benchmark excluded) -> diversification candidates
pairs = corr_matrix.loc[assets_only, assets_only].where(
    np.triu(np.ones((len(assets_only),) * 2), k=1).astype(bool)).stack().sort_values()
print("Lowest-correlation pairs (best diversification candidates):")
display(pairs.rename("correlation").to_frame().head(3))
print("Highest-correlation pairs (least diversification):")
display(pairs.rename("correlation").to_frame().tail(3))
print("Correlation with benchmark:")
display(corr_matrix.loc[assets_only, bench].rename("corr_with_benchmark").to_frame())

**Interpretation (Observation - numerical evidence - financial implication - limitation)**

> **TODO: PROMPT for Claude:** *"Here are my covariance and correlation matrices and the ranked pairs: [paste]. Write the interpretation in the sequence Observation - numerical evidence - financial implication - limitation. State the relationship of each asset with the benchmark and which pairs may be useful for diversification. Note that correlation is not causation and that it can change over time."*

### 4.3 Findings the Investment Committee could act upon (minimum three)

> **TODO: PROMPT for Claude:** *"From my tables, write three (or more) evidence-based findings, each in the form: Finding - statistic that supports it - what the Committee could do (monitor / investigate further). Do not present them as personal investment advice."*
>
> 1. TODO
> 2. TODO
> 3. TODO

---
## Task 5 - Decision Dashboard and Visual Interpretation
Each figure has a decision-focused title, axis labels, legend, source/period note, and is exported to `outputs/figures/`.

In [ ]:
period_note = (f"Source: {SOURCE_NAME}; retrieved {RETRIEVAL_DATE}; period "
               f"{prices.index.min():%d %b %Y} - {prices.index.max():%d %b %Y}; price field: {PRICE_FIELD}.")
bench = list(BENCHMARK)[0]

### Figure 1 - Normalised price index (start = 100)

In [ ]:
# TODO(PROMPT): Replace the title with a decision-focused one that states the answer, e.g.
# "Only two of four assets outperformed the benchmark over the period"
fig1 = ft.plot_normalized(normalized, bench,
    title="Which assets outperformed the benchmark? Growth of 100 invested at the start",
    source_note=period_note, save_path=FIG_DIR / "fig1_normalized_index.png")
plt.show()

**Interpretation (2-3 sentences: what happened - evidence in the figure - why it matters)**

> **TODO: PROMPT:** *"Look at Figure 1 and my summary table [paste]. Write 2-3 sentences: what happened, the evidence visible in the chart (end index levels, periods of divergence), and why it matters to the Investment Committee."*

### Figure 2 - Return distributions

In [ ]:
# TODO(PROMPT): Replace the title with a decision-focused one (e.g. which asset has fat tails?)
fig2 = ft.plot_histograms(returns,
    title="Which return distributions show heavy tails or asymmetric risk?",
    source_note=period_note, save_path=FIG_DIR / "fig2_return_distributions.png")
plt.show()

**Interpretation**

> **TODO: PROMPT:** *"Using skewness and excess kurtosis from my table [paste] and Figure 2, write 2-3 sentences: what the distributions show, the evidence, and why tail risk matters for monitoring."*

### Figure 3 - Risk-return chart

In [ ]:
# TODO(PROMPT): Replace the title with a decision-focused one (e.g. which asset offers the best return per unit of risk?)
fig3 = ft.plot_risk_return(returns,
    title="Which assets compensate investors for higher daily risk?",
    source_note=period_note, save_path=FIG_DIR / "fig3_risk_return.png")
plt.show()

**Interpretation**

> **TODO: PROMPT:** *"Write 2-3 sentences on Figure 3: which points sit high/left (more return for less risk) and low/right, the numbers behind it, and why it matters. Remember the axes are mean DAILY return and daily standard deviation in %."*

### Figure 4 - Correlation heatmap

In [ ]:
# TODO(PROMPT): Replace the title with a decision-focused one (e.g. where is diversification available?)
fig4 = ft.plot_heatmap(corr_matrix,
    title="Where can diversification be found? Correlation of daily returns",
    source_note=period_note, save_path=FIG_DIR / "fig4_correlation_heatmap.png")
plt.show()

**Interpretation**

> **TODO: PROMPT:** *"Write 2-3 sentences on Figure 4: strongest and weakest correlations, relationship with the benchmark, and what it means for diversification and monitoring."*

In [ ]:
# Final verification: every deliverable file exists
expected = [RAW_FILE, CLEAN_FILE, AUDIT_FILE, QUALITY_LOG_FILE, SUMMARY_FILE, COV_FILE, CORR_FILE,
            FIG_DIR / "fig1_normalized_index.png", FIG_DIR / "fig2_return_distributions.png",
            FIG_DIR / "fig3_risk_return.png", FIG_DIR / "fig4_correlation_heatmap.png"]
display(ft.check_files_exist(expected))

---
## Conclusion, limitations and declarations

> **TODO: PROMPT for Claude:** *"Write a 120-word conclusion and a limitations paragraph for this analysis (data source, period length, daily returns only, correlation instability, no transaction costs, no portfolio optimisation or forecasting)."*

**References**
- pandas documentation - https://pandas.pydata.org/docs/
- NumPy documentation - https://numpy.org/doc/
- Matplotlib documentation - https://matplotlib.org/stable/
- yfinance documentation - https://ranaroussi.github.io/yfinance/
- TODO: add the exact dataset reference (source, URL, retrieval date) and any other sources.

**AI-assistance declaration:** TODO - state exactly how AI tools were used, following your lecturer's instructions. Every member must understand and be able to defend all code.